# Blue Archive ガチャシミュレーション — レビューしやすい簡潔版

構成は次の4つだけです。

1. `Config`：設定
2. `State`：シミュレーション中の状態
3. `draw_one` / `draw_batch`：抽選
4. `simulate_once` / `simulate_many`：制御

終了条件とPU1→PU2切替条件は、どちらも `State -> bool` の関数です。


In [1]:
from __future__ import annotations

from dataclasses import dataclass
from enum import Enum, auto
from typing import Callable
import random
import statistics

import matplotlib.pyplot as plt


## 1. 設定と状態

In [2]:
class Banner(Enum):
    PU1 = auto()
    PU2 = auto()


class Draw(Enum):
    STAR1 = auto()
    STAR2 = auto()
    STAR3_OTHER = auto()
    PU1 = auto()
    PU2 = auto()


@dataclass(frozen=True)
class Config:
    pickup_rate: float = 0.007
    other_pickup_rate: float = 0.00022115
    star3_rate: float = 0.03
    star2_or_higher_rate: float = 0.215

    initial_charge: int = 0
    initial_banner: Banner = Banner.PU1
    pulls_per_batch: int = 10

    charge_100_pickup_rate: float = 0.5
    charge_100: int = 100
    charge_200: int = 200

    def __post_init__(self):
        if not 0 <= self.pickup_rate <= 1:
            raise ValueError("pickup_rate must be between 0 and 1.")
        if not 0 <= self.other_pickup_rate <= 1:
            raise ValueError("other_pickup_rate must be between 0 and 1.")
        if self.pickup_rate + self.other_pickup_rate > self.star3_rate:
            raise ValueError(
                "pickup_rate + other_pickup_rate must not exceed star3_rate."
            )
        if self.star3_rate > self.star2_or_higher_rate:
            raise ValueError(
                "star3_rate must not exceed star2_or_higher_rate."
            )
        if not 0 <= self.initial_charge < self.charge_200:
            raise ValueError(
                "initial_charge must satisfy 0 <= initial_charge < charge_200."
            )
        if self.pulls_per_batch <= 0:
            raise ValueError("pulls_per_batch must be positive.")


@dataclass
class State:
    pulls: int = 0
    batches: int = 0
    charge: int = 0
    banner: Banner = Banner.PU1

    pu1: int = 0
    pu2: int = 0
    star3: int = 0

    pu1_banner_pulls: int = 0
    pu2_banner_pulls: int = 0


## 2. 抽選

In [3]:
def target_draw(banner: Banner) -> Draw:
    return Draw.PU1 if banner is Banner.PU1 else Draw.PU2


def other_pickup_draw(banner: Banner) -> Draw:
    return Draw.PU2 if banner is Banner.PU1 else Draw.PU1


def draw_one(config: Config, state: State, rng: random.Random) -> Draw:
    next_charge = state.charge + 1

    if next_charge == config.charge_200:
        return target_draw(state.banner)

    if next_charge == config.charge_100:
        if rng.random() < config.charge_100_pickup_rate:
            return target_draw(state.banner)
        return other_pickup_draw(state.banner)

    x = rng.random()

    if x < config.pickup_rate:
        return target_draw(state.banner)
    if x < config.pickup_rate + config.other_pickup_rate:
        return other_pickup_draw(state.banner)
    if x < config.star3_rate:
        return Draw.STAR3_OTHER
    if x < config.star2_or_higher_rate:
        return Draw.STAR2
    return Draw.STAR1


In [4]:
def apply_draw(state: State, result: Draw) -> None:
    state.pulls += 1
    state.charge += 1

    if state.banner is Banner.PU1:
        state.pu1_banner_pulls += 1
    else:
        state.pu2_banner_pulls += 1

    if result in {Draw.STAR3_OTHER, Draw.PU1, Draw.PU2}:
        state.star3 += 1

    if result is Draw.PU1:
        state.pu1 += 1
    elif result is Draw.PU2:
        state.pu2 += 1

    current_banner_pickup = (
        state.banner is Banner.PU1 and result is Draw.PU1
    ) or (
        state.banner is Banner.PU2 and result is Draw.PU2
    )

    if current_banner_pickup:
        state.charge = 0


def draw_batch(config: Config, state: State, rng: random.Random) -> None:
    state.batches += 1

    for _ in range(config.pulls_per_batch):
        result = draw_one(config, state, rng)
        apply_draw(state, result)


## 3. 条件関数

In [5]:
Condition = Callable[[State], bool]


def stop_after_pulls(max_pulls: int) -> Condition:
    return lambda state: state.pulls >= max_pulls


def stop_when_both_obtained(state: State) -> bool:
    return state.pu1 >= 1 and state.pu2 >= 1


def switch_when_pu1_obtained(state: State) -> bool:
    return state.pu1 >= 1


def switch_after_pu1_pulls(max_pulls: int) -> Condition:
    return lambda state: state.pu1_banner_pulls >= max_pulls


def switch_when_pu1_or_limit(max_pulls: int) -> Condition:
    return lambda state: (
        state.pu1 >= 1
        or state.pu1_banner_pulls >= max_pulls
    )


def never(_: State) -> bool:
    return False


## 4. シミュレーション

判定順序は次の通りです。

1. `pulls_per_batch` 回をすべて引く
2. 終了条件を評価
3. 終了しない場合だけPU1→PU2切替条件を評価

したがって10連の途中でPU1を引いても、残りはPU1募集を引きます。


In [6]:
def simulate_once(
    config: Config,
    rng: random.Random,
    stop_condition: Condition,
    switch_condition: Condition = never,
) -> State:
    state = State(
        charge=config.initial_charge,
        banner=config.initial_banner,
    )

    while not stop_condition(state):
        draw_batch(config, state, rng)

        if stop_condition(state):
            break

        if state.banner is Banner.PU1 and switch_condition(state):
            state.banner = Banner.PU2

    return state


def simulate_many(
    config: Config,
    n_simulations: int,
    stop_condition: Condition,
    switch_condition: Condition = never,
    seed: int = 42,
) -> list[State]:
    if n_simulations <= 0:
        raise ValueError("n_simulations must be positive.")

    rng = random.Random(seed)

    return [
        simulate_once(
            config=config,
            rng=rng,
            stop_condition=stop_condition,
            switch_condition=switch_condition,
        )
        for _ in range(n_simulations)
    ]


## 5. 集計

In [7]:
def summarize(results: list[State]) -> dict[str, float]:
    if not results:
        raise ValueError("results must not be empty.")

    n = len(results)

    return {
        "PU1入手率": sum(r.pu1 >= 1 for r in results) / n,
        "PU2入手率": sum(r.pu2 >= 1 for r in results) / n,
        "両方入手率": sum(
            r.pu1 >= 1 and r.pu2 >= 1
            for r in results
        ) / n,
        "平均PU1人数": statistics.fmean(r.pu1 for r in results),
        "平均PU2人数": statistics.fmean(r.pu2 for r in results),
        "平均★3人数": statistics.fmean(r.star3 for r in results),
        "平均連数": statistics.fmean(r.pulls for r in results),
        "平均バッチ数": statistics.fmean(r.batches for r in results),
        "平均PU1募集連数": statistics.fmean(
            r.pu1_banner_pulls for r in results
        ),
        "平均PU2募集連数": statistics.fmean(
            r.pu2_banner_pulls for r in results
        ),
    }


def print_summary(summary: dict[str, float]) -> None:
    for name, value in summary.items():
        if "率" in name:
            print(f"{name}: {value:.4%}")
        else:
            print(f"{name}: {value:.4f}")


## 6. 実行例

In [ ]:
config = Config(
    pickup_rate=0.007,
    other_pickup_rate=0.00022115,
    star3_rate=0.03,
    star2_or_higher_rate=0.215,
    initial_charge=0,
    initial_banner=Banner.PU1,
    pulls_per_batch=10,
)

results = simulate_many(
    config=config,
    n_simulations=100_000,
    seed=42,

    stop_condition=lambda state: (
        stop_when_both_obtained(state)
        or state.pulls >= 400
    ),

    switch_condition=switch_when_pu1_obtained,
)

print_summary(summarize(results))


In [ ]:
plt.figure(figsize=(9, 5))
plt.hist([r.pulls for r in results], bins=40)
plt.xlabel("Total pulls")
plt.ylabel("Frequency")
plt.title("Distribution of total pulls")
plt.tight_layout()
plt.show()


## 7. 簡易テスト

In [ ]:
def run_tests():
    config = Config(
        pickup_rate=1.0,
        other_pickup_rate=0.0,
        star3_rate=1.0,
        star2_or_higher_rate=1.0,
        pulls_per_batch=10,
    )

    result = simulate_once(
        config=config,
        rng=random.Random(1),
        stop_condition=stop_when_both_obtained,
        switch_condition=switch_when_pu1_obtained,
    )

    assert result.pu1_banner_pulls == 10
    assert result.pu2_banner_pulls == 10
    assert result.pu1 == 10
    assert result.pu2 == 10
    assert result.pulls == 20
    assert result.batches == 2

    print("All tests passed.")


run_tests()


## 8. 切替条件の例

```python
# PU1を引いたらPU2へ
switch_condition = switch_when_pu1_obtained

# PU1募集を100連引いたらPU2へ
switch_condition = switch_after_pu1_pulls(100)

# PU1を引くか100連に達したらPU2へ
switch_condition = switch_when_pu1_or_limit(100)

# 独自条件
def custom_switch(state):
    return (
        state.pu1 >= 2
        or state.pu1_banner_pulls >= 150
        or state.star3 >= 8
    )
```


In [8]:
### 実行テスト
config = Config(
    pickup_rate=0.007,
    other_pickup_rate=0.00022115,
    star3_rate=0.03,
    star2_or_higher_rate=0.215,
    initial_charge=0,
    initial_banner=Banner.PU1,
    pulls_per_batch=10,
)


In [11]:
results = simulate_many(
    config=config,
    n_simulations=2,
    seed=42,

    stop_condition=lambda state: (
        stop_when_both_obtained(state)
        or state.pulls >= 400
    ),

    switch_condition=switch_when_pu1_obtained,
)

In [12]:
results

[State(pulls=120, batches=12, charge=0, banner=<Banner.PU2: 2>, pu1=1, pu2=1, star3=6, pu1_banner_pulls=20, pu2_banner_pulls=100),
 State(pulls=110, batches=11, charge=5, banner=<Banner.PU2: 2>, pu1=1, pu2=1, star3=3, pu1_banner_pulls=10, pu2_banner_pulls=100)]

In [13]:
config=config
n_simulations=2
seed=42

stop_condition=lambda state: (
    stop_when_both_obtained(state)
    or state.pulls >= 400
)

switch_condition=switch_when_pu1_obtained

if n_simulations <= 0:
    raise ValueError("n_simulations must be positive.")

rng = random.Random(seed)

result = [
    simulate_once(
        config=config,
        rng=rng,
        stop_condition=stop_condition,
        switch_condition=switch_condition,
    )
    for _ in range(n_simulations)
]

In [14]:
result

[State(pulls=120, batches=12, charge=0, banner=<Banner.PU2: 2>, pu1=1, pu2=1, star3=6, pu1_banner_pulls=20, pu2_banner_pulls=100),
 State(pulls=110, batches=11, charge=5, banner=<Banner.PU2: 2>, pu1=1, pu2=1, star3=3, pu1_banner_pulls=10, pu2_banner_pulls=100)]

In [15]:
config=config
rng=rng
stop_condition=stop_condition
switch_condition=switch_condition
switch_condition

state = State(
    charge=config.initial_charge,
    banner=config.initial_banner,
)

while not stop_condition(state):
    draw_batch(config, state, rng)
    if stop_condition(state):
        break
    if state.banner is Banner.PU1 and switch_condition(state):
        state.banner = Banner.PU2

state

State(pulls=140, batches=14, charge=1, banner=<Banner.PU2: 2>, pu1=1, pu2=1, star3=6, pu1_banner_pulls=70, pu2_banner_pulls=70)